# Figure 4: Demographic effects on B cell repertoire maturation and isotype usage

This notebook reproduces panels of **Figure 4** of the AIDA AIRR manuscript:

- **Fig. 4A** — UMAP of B cells colored by subset
- **Fig. 4B** — UMAP of B cells colored by somatic hypermutation (SHM) rate
- **Fig. 4C** — UMAP of B cells colored by isotype subclass
- **Fig. 4D** — Per-donor Gini coefficient vs. age in memory B and plasma cells (with rarefied inset)
- **Fig. 4E** — Accumulation of SHM with age across B cell subsets
- **Fig. 4F** — Age-associated shifts in isotype composition within memory B cells
- **Fig. 4G** — Ridge regression heatmap linking demographic factors to BCR features
  (clonality, SHM, isotype usage)


## Imports

In [ ]:
import warnings
warnings.filterwarnings(action='ignore')

import numpy as np
import pandas as pd
import scipy as sp
import scipy.stats as stats
from scipy.stats import norm
from scipy.sparse import csr_matrix

import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects
import seaborn as sns

from mpl_toolkits.axes_grid1.inset_locator import inset_axes

import scanpy as sc
import anndata as ad
import dandelion as ddl
import scirpy as ir
import sceleto2 as scjp

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler as standard
from sklearn.utils import resample
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multitest import multipletests

%matplotlib inline
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=100, color_map='OrRd')

plt.rcParams['pdf.fonttype'] = 42
sns.set_style('ticks', {'axes.edgecolor': 'black', 'axes.edgewidth': 2})
sns.set_context("paper", font_scale=1.3, rc={'patch.linewidth': 1})


## Color palettes / category orders

In [ ]:
Ethnicity_order = ['Chinese', 'Malay', 'Indian', 'Japanese', 'Korean', 'Thai']
Ethnicity_colors = ["tomato", 'gold', "dodgerblue", "limegreen", "aquamarine", "orchid", 'gray']
sex_colors = ["#87CEFA", "#FB7C7C"]

iso_order = ['IGHD', 'IGHM', 'IGHG1', 'IGHG2', 'IGHG3', 'IGHG4', 'IGHA1', 'IGHA2', 'IGHE']
iso_color = ['slategrey', 'wheat', 'lightskyblue', 'dodgerblue', 'navy', 'teal',
             'palevioletred', 'lightsalmon', 'silver']

B_anno2_order = [
    'B_Naive', 'B_Memory_USW', 'B_Memory_SW', 'B_Memory_Atypical',
    'B_Plasma',
]


## Load B cell data and assemble metadata

In [ ]:
bdata = sc.read('data/06_250608_BDATA_IGLK_annotated_meta.h5ad')
sns.histplot(bdata.obs['PatientID'].value_counts(),bins=500)
plt.xlim(0,200)

In [ ]:
bdata = bdata[bdata.obs['j_call_VJ_main']!='No_contig']

In [ ]:
# QC: exclude low-cell-count donors (5th percentile)
import pandas as pd
pco = pd.DataFrame(bdata.obs['PatientID'].value_counts())
ptl = pco[pco['count'] >= np.percentile(pco['count'], 5)].index.tolist()


In [ ]:
adata = bdata[bdata.obs['PatientID'].isin(ptl)]

adata = adata[~adata.obs['Ethnicity'].isna()]

In [ ]:
adata = adata[adata.obs['Ethnicity']!='European']

In [ ]:
adata.obs['v_call_B_VDJ_main_old'] = adata.obs['v_call_B_VDJ_main'].copy()
adata.obs['v_call_B_VDJ_main'] = [a.split(',')[0] for a in adata.obs['v_call_B_VDJ_main_old']]

adata.obs['j_call_B_VDJ_main_old'] = adata.obs['j_call_B_VDJ_main'].copy()
adata.obs['j_call_B_VDJ_main'] = [a.split(',')[0] for a in adata.obs['j_call_B_VDJ_main']]

adata.obs['v_call_B_VJ_main_old'] = adata.obs['v_call_B_VJ_main'].copy()
adata.obs['v_call_B_VJ_main'] = [a.split(',')[0] for a in adata.obs['v_call_B_VJ_main_old']]

adata.obs['j_call_B_VJ_main_old'] = adata.obs['j_call_B_VJ_main'].copy()
adata.obs['j_call_B_VJ_main'] = [a.split(',')[0] for a in adata.obs['j_call_B_VJ_main_old']]

In [ ]:
adata.obs.v_call_B_VDJ_main = np.where(adata.obs['v_call_B_VDJ_main'].isin(['IGHV3-23D']), 'IGHV3-23',  adata.obs['v_call_B_VDJ_main'])
adata.obs.v_call_B_VDJ_main = np.where(adata.obs['v_call_B_VDJ_main'].isin(['IGHV3-30-3', 'IGHV3-30-5', 'IGHV3-33']), 'IGHV3-30',  adata.obs['v_call_B_VDJ_main'])
adata.obs.v_call_B_VDJ_main = np.where(adata.obs['v_call_B_VDJ_main'].isin(['IGHV1-69D']), 'IGHV1-69',  adata.obs['v_call_B_VDJ_main'])
adata.obs.v_call_B_VDJ_main = np.where(adata.obs['v_call_B_VDJ_main'].isin(['IGHV1-69-2']), 'IGHV1-69',  adata.obs['v_call_B_VDJ_main'])
adata.obs.v_call_B_VDJ_main = np.where(adata.obs['v_call_B_VDJ_main'].isin(['IGHV3-43D']), 'IGHV3-43',  adata.obs['v_call_B_VDJ_main'])
adata.obs.v_call_B_VDJ_main = np.where(adata.obs['v_call_B_VDJ_main'].isin(['IGHV3-64D']), 'IGHV3-64',  adata.obs['v_call_B_VDJ_main'])
adata.obs.v_call_B_VDJ_main = np.where(adata.obs['v_call_B_VDJ_main'].isin(['IGHV3-66']), 'IGHV3-53',  adata.obs['v_call_B_VDJ_main'])
adata.obs.v_call_B_VDJ_main = np.where(adata.obs['v_call_B_VDJ_main'].isin(['IGHV4-30-2']), 'IGHV4-30-4',  adata.obs['v_call_B_VDJ_main'])

In [ ]:
adata.obs['anno2'] = np.where(adata.obs['anno2']=='B_Plasmablast', 'B_Plasma', adata.obs['anno2'])

In [ ]:
adata.obs['anno2'] =adata.obs['anno2'].astype('category')

Per-donor metadata (used by all panels):

In [ ]:
meta = adata.obs.drop_duplicates('PatientID').set_index('PatientID')[['Age', 'Sex', 'BMI', 'Ethnicity']]


## Compute per-(donor x subset) Gini coefficient

In [ ]:
import scirpy as ir

In [ ]:
vdata= adata.copy()

In [ ]:
vdata.obs['Pt_anno1'] = ['{}*{}'.format(a,b) for a,b in zip(vdata.obs['PatientID'],vdata.obs['anno1'])]

In [ ]:
vdata.obs['clone_id_Pt'] = ['{}*{}'.format(a,b) for a, b in zip(vdata.obs['clone_id'],
                                                                vdata.obs['PatientID'])]

In [ ]:
meta = vdata.obs.drop_duplicates('PatientID').set_index('PatientID')[['Age', 'Sex', 'BMI', 'Ethnicity']]


In [ ]:
ir.tl.alpha_diversity(
    vdata, groupby='Pt_anno1', target_col='clone_id_Pt', key_added='D50_anno1'
)

In [ ]:
def gini_index(array):
    """
    Compute the Gini coefficient of an array.
    :param array: numeric array representing a distribution
    :return: Gini coefficient (between 0 and 1)
    """
    # sort the array
    array = np.sort(array)
    index = np.arange(1, len(array) + 1)
    n = len(array)
    
    ## compute Gini coefficient
    return ((2 * np.sum(index * array)) / (n * np.sum(array))) - ((n + 1) / n)

In [ ]:
# Compute per-cell Gini coefficient using clone size distributions per (PatientID, anno1)
vdata.obs['Pt_anno1'] = vdata.obs['Pt_anno1'].astype('str')
vdata.obs['clone_id_Pt'] = vdata.obs['clone_id_Pt'].astype('str')

gini_dict = {}
for pt_anno1, sub in vdata.obs.groupby('Pt_anno1'):
    counts = sub['clone_id_Pt'].value_counts().values
    if len(counts) == 0 or counts.sum() == 0:
        gini_dict[pt_anno1] = np.nan
    else:
        gini_dict[pt_anno1] = gini_index(counts)

vdata.obs['gini_anno1'] = vdata.obs['Pt_anno1'].map(gini_dict)


### Fig. 4A — UMAP of B cells colored by subset (`anno2`)

In [ ]:
scjp.us(vdata, 'anno2', frameon=False, legend_loc='on data', legend_fontsize=9, size=2)
plt.title('')
scjp.save_fig('Fig4_', 'B_UMAP_anno2', fig_folder='figures')


### Per-cell Gini coefficient overlay UMAP (Fig. 4 supplement)

In [ ]:
scjp.us(vdata, 'gini_anno1', vmax=0.36, 
        size=5, frameon=False)
plt.title('Gini coefficient index')

scjp.save_fig('Fig5_','B_Gini_UMAP',fig_folder='figures')

## Fig. 4D — Per-donor Gini coefficient vs. age in memory B and plasma cells

In [ ]:
g0 = vdata.obs.groupby('Pt_anno1').mean()[['gini_anno1']]

g0['PatientID'] = [a.split('*')[0] for a in g0.index]
g0['anno1'] = [a.split('*')[1] for a in g0.index]


In [ ]:
poco = vdata.obs['Pt_anno1'].value_counts()[vdata.obs['Pt_anno1'].value_counts()>=10].index.tolist()

In [ ]:
g0 = g0[g0.index.isin(poco)]

In [ ]:
g0 = g0.merge(meta.reset_index(), on='PatientID', how='left')

plt.figure(figsize=(4,4))
sns.barplot(data=g0, x='anno1', y='gini_anno1', hue='Ethnicity', 
          order= ['B_Memory', 'B_Plasma'],
           hue_order=Ethnicity_order, palette=Ethnicity_colors, 
            alpha=0.8,
            errcolor='black', ec='black', errwidth=0.5,  errorbar='se', linewidth=0.1,
capsize=0.02,)
plt.xticks(rotation=0)
plt.xlabel('')
plt.ylabel('Gini Coefficient')
plt.legend(loc=(1.01,0), edgecolor='k')
sns.despine()
plt.show()

### Per-donor Gini vs age scatterplots, with inset of donors with 0<Gini<X

In [ ]:
for cell in ['B_Memory','B_Plasma']:
    # 1) create main plot
    fig, ax = plt.subplots(figsize=(4,5), dpi=100)
    sub = g0[g0.anno1 == cell].dropna(subset='gini_anno1')
    
    # scatter + regression line
    sns.scatterplot(
        data=sub, x='Age', y='gini_anno1', hue='Sex',
        hue_order=['Male','Female'], palette=sex_colors,
        ax=ax, ec='black', s=30, alpha=0.7
    )
    sns.regplot(
        data=sub[sub.Sex=='Male'], x='Age', y='gini_anno1',
        scatter=False, ax=ax, 
        line_kws={'lw':3, 'ls':'--', 'alpha':0.8, 'color':sex_colors[0]}
    )
    sns.regplot(
        data=sub[sub.Sex=='Female'], x='Age', y='gini_anno1',
        scatter=False, ax=ax,
        line_kws={'lw':3, 'ls':'--', 'alpha':0.8,'color':sex_colors[1]}
    )
    # add black outline effect to the line
    for line in ax.lines:
        line.set_path_effects([
            path_effects.Stroke(linewidth=5, foreground='black'),
            path_effects.Normal()
        ])
    
    # compute r, p; set title/labels
    r_value, p_value = stats.pearsonr(sub['Age'], sub['gini_anno1'])
    ax.set_title(f"{cell} (r={r_value:.3f}, p={p_value:.3f})", fontsize=16, color='k')
    ax.set_xlabel('Age', fontsize=14)
    ax.set_ylabel('Gini coefficient index', fontsize=14)
    sns.despine()
    # 2) create inset axes (top-left, 40% x 30%)
    axins = inset_axes(ax, width="50%", height="50%", loc='upper left', borderpad=2)
    sns.scatterplot(
        data=sub, x='Age', y='gini_anno1', hue='Sex',
        hue_order=['Male','Female'], palette=sex_colors,
        ax=axins, legend=False, ec='black', s=30, alpha=0.7
    )
    sns.regplot(
        data=sub[sub.Sex=='Male'], x='Age', y='gini_anno1',
        scatter=False, ax=axins, 
        line_kws={'lw':3, 'ls':'--', 'alpha':0.8,'color':sex_colors[0]}
    )
    sns.regplot(
        data=sub[sub.Sex=='Female'], x='Age', y='gini_anno1',
        scatter=False, ax=axins,
        line_kws={'lw':3, 'ls':'--', 'alpha':0.8,'color':sex_colors[1]}
    )
    for line in axins.lines:
        line.set_path_effects([
            path_effects.Stroke(linewidth=5, foreground='black'),
            path_effects.Normal()
        ])
    # set zoom region for inset axes
    axins.set_xlim(ax.get_xlim()[0], ax.get_xlim()[1])
    axins.set_ylim(0, 0.2)
    axins.set_xticks([])
    axins.set_yticks([0,0.2])
    axins.set_ylabel('')
    axins.set_xlabel('')
    axins.set_title('', fontsize=10)
    axins.spines['bottom'].set_linewidth(1)
    axins.spines['left'].set_linewidth(1)
    axins.spines['top'].set_linewidth(1)
    axins.spines['right'].set_linewidth(1)
    
    # 3) save final figure
    plt.tight_layout()
    
    fig.savefig(f'figures/Fig5_gini_{cell}_inset.pdf',
                dpi=300, format='pdf', transparent=True, bbox_inches='tight')
    plt.show()

### Rarefied Gini vs age (controls for sampling-depth dependence)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.patheffects as path_effects

from scipy import stats
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# -----------------------------
# settings
# -----------------------------
ptanno_col = "Pt_anno1"
clone_col = "clone_id"   # <- modify if needed

raref_depth_map = {
    "B_Memory": 100,   # adjust if needed
    "B_Plasma": 10,   # adjust if needed
}

target_cells = ["B_Memory", "B_Plasma"]
n_repeat_raref = 200

# -----------------------------
# helper
# -----------------------------
def gini_from_counts(counts):
    x = np.asarray(counts, dtype=float)
    if len(x) == 0 or np.sum(x) == 0:
        return np.nan
    x = np.sort(x)
    n = len(x)
    return (2 * np.sum(np.arange(1, n + 1) * x)) / (n * np.sum(x)) - (n + 1) / n

def rarefied_gini(clone_ids, depth, n_repeat=200, random_state=0):
    clone_ids = np.asarray(clone_ids)
    n_total = len(clone_ids)
    if n_total < depth:
        return np.nan

    rng = np.random.default_rng(random_state)
    vals = []
    for i in range(n_repeat):
        idx = rng.choice(n_total, size=depth, replace=False)
        sampled = clone_ids[idx]
        counts = pd.Series(sampled).value_counts().values
        vals.append(gini_from_counts(counts))
    return np.mean(vals)

# -----------------------------
# 1) keep only usable cells
# -----------------------------
tmp = vdata.obs[[ptanno_col, clone_col]].copy()
tmp = tmp.dropna(subset=[ptanno_col, clone_col]).copy()

tmp[ptanno_col] = tmp[ptanno_col].astype(str)
tmp[clone_col] = tmp[clone_col].astype(str)

tmp = tmp[
    (~tmp[clone_col].isin(["", "nan", "None", "NA"])) &
    (tmp[clone_col].str.strip() != "")
].copy()

tmp["PatientID"] = tmp[ptanno_col].str.split("*").str[0]
tmp["anno1"] = tmp[ptanno_col].str.split("*").str[1]

tmp = tmp[tmp["anno1"].isin(target_cells)].copy()

# -----------------------------
# 2) compute rarefied Gini per Pt_anno1
# -----------------------------
rows = []

for ptanno, sub in tmp.groupby(ptanno_col):
    cell = sub["anno1"].iloc[0]
    depth = raref_depth_map[cell]
    n_cells = len(sub)

    if n_cells < depth:
        continue

    rg = rarefied_gini(
        clone_ids=sub[clone_col].values,
        depth=depth,
        n_repeat=n_repeat_raref,
        random_state=0
    )

    rows.append({
        "Pt_anno1": ptanno,
        "PatientID": sub["PatientID"].iloc[0],
        "anno1": cell,
        "raref_depth": depth,
        "n_cells_anno1": n_cells,
        "gini_rarefied": rg
    })

g0r = pd.DataFrame(rows)

# apply the existing poco filter if needed
# poco = vdata.obs['Pt_anno1'].value_counts()[vdata.obs['Pt_anno1'].value_counts() >= 10].index.tolist()
# g0r = g0r[g0r['Pt_anno1'].isin(poco)].copy()

# meta merge
g0r = g0r.merge(meta.reset_index(), on="PatientID", how="left")


In [ ]:

# -----------------------------
# 3) plot
# -----------------------------
for cell in target_cells:
    fig, ax = plt.subplots(figsize=(4, 5), dpi=100)

    sub = g0r[g0r.anno1 == cell].copy()
    sub = sub.dropna(subset=["Age", "Sex", "gini_rarefied"])

    if sub.shape[0] < 3:
        print(f"skip {cell}: too few samples")
        continue

    # main scatter + reg lines
    sns.scatterplot(
        data=sub, x="Age", y="gini_rarefied", hue="Sex",
        hue_order=["Male", "Female"], palette=sex_colors,
        ax=ax, ec="black", s=30, alpha=0.7
    )
    sns.regplot(
        data=sub[sub.Sex == "Male"], x="Age", y="gini_rarefied",
        scatter=False, ax=ax,
        line_kws={"lw": 3, "ls": "--", "alpha": 0.8, "color": sex_colors[0]}
    )
    sns.regplot(
        data=sub[sub.Sex == "Female"], x="Age", y="gini_rarefied",
        scatter=False, ax=ax,
        line_kws={"lw": 3, "ls": "--", "alpha": 0.8, "color": sex_colors[1]}
    )

    for line in ax.lines:
        line.set_path_effects([
            path_effects.Stroke(linewidth=5, foreground="black"),
            path_effects.Normal()
        ])

    r_value, p_value = stats.pearsonr(sub["Age"], sub["gini_rarefied"])
    ax.set_title(f"{cell} (d={raref_depth_map[cell]})\nr={r_value:.3f}, p={p_value:.3f}",
                 fontsize=16, color="k")
    ax.set_xlabel("Age", fontsize=14)
    ax.set_ylabel("Rarefied Gini coefficient", fontsize=14)
    sns.despine(ax=ax)

    # inset
    axins = inset_axes(ax, width="50%", height="50%", loc="upper left", borderpad=2)

    sns.scatterplot(
        data=sub, x="Age", y="gini_rarefied", hue="Sex",
        hue_order=["Male", "Female"], palette=sex_colors,
        ax=axins, legend=False, ec="black", s=30, alpha=0.7
    )
    sns.regplot(
        data=sub[sub.Sex == "Male"], x="Age", y="gini_rarefied",
        scatter=False, ax=axins,
        line_kws={"lw": 3, "ls": "--", "alpha": 0.8, "color": sex_colors[0]}
    )
    sns.regplot(
        data=sub[sub.Sex == "Female"], x="Age", y="gini_rarefied",
        scatter=False, ax=axins,
        line_kws={"lw": 3, "ls": "--", "alpha": 0.8, "color": sex_colors[1]}
    )

    for line in axins.lines:
        line.set_path_effects([
            path_effects.Stroke(linewidth=5, foreground="black"),
            path_effects.Normal()
        ])

    axins.set_xlim(ax.get_xlim()[0], ax.get_xlim()[1])


        # set zoom region for inset axes
    axins.set_xlim(ax.get_xlim()[0], ax.get_xlim()[1])
    axins.set_ylim(0, 0.18)
    axins.set_xticks([])
    axins.set_yticks([0,0.18])
    axins.set_ylabel('')
    axins.set_xlabel('')
    axins.set_title('', fontsize=10)
    axins.spines['bottom'].set_linewidth(1)
    axins.spines['left'].set_linewidth(1)
    axins.spines['top'].set_linewidth(1)
    axins.spines['right'].set_linewidth(1)

    for spine in ["bottom", "left", "top", "right"]:
        axins.spines[spine].set_linewidth(1)

    plt.tight_layout()
    fig.savefig(
        f'figures/Fig5_gini_rarefied_{cell}_inset.pdf',
        dpi=300, format='pdf', transparent=True, bbox_inches='tight'
    )
    plt.show()

## Fig. 4B / 4E — Somatic hypermutation overview and age dependence

### Fig. 4B — UMAP of SHM rate (`mu_freq`)

In [ ]:
crim = sns.color_palette("light:crimson", as_cmap=True)

In [ ]:
scjp.us(vdata, 'mu_freq', size=5, frameon=False)
plt.title('Somatic hypermutation rate')
scjp.save_fig('Fig4_', 'B_SHM_UMAP', fig_folder='figures')


### Per-donor SHM x cell type stratified by ethnicity (age 30-60)

In [ ]:
vmf = vdata.obs.groupby(['PatientID','anno2']).mean()[['mu_freq']].reset_index()

vmf = vmf[~vmf.anno2.str.endswith('IFN')]
vmf = vmf[~vmf.anno2.str.startswith('B_N')]

vmf = vmf.merge(meta, on='PatientID')

In [ ]:
vmfe = vmf[(30<=vmf['Age']) & (vmf['Age']<60)]

vmfe['anno2'] = vmfe['anno2'].astype('str')

plt.figure(figsize=(7,5))

g = sns.barplot(data=vmfe, x='anno2', y='mu_freq', hue='Ethnicity',
             palette=Ethnicity_colors, hue_order=Ethnicity_order, saturation=0.8,errorbar='se',
            errcolor='black', ec='black', errwidth=1,  
capsize=0.05,)
# g.set_ylim([0,0.6])
#plt.title('vmfetype proportion in memory B cell')
plt.xlabel('')
plt.ylabel('Proportion')
sns.despine()
# g.get_legend().remove()
plt.legend(loc='upper left', edgecolor='black')
plt.xticks(rotation=30)

    

for i,f in enumerate(vmfe.anno2.unique()):
    df1=vmfe[vmfe['anno2']==f]
    df1['Sex'] = df1['Sex'].map({'Male': 0, 'Female': 1})
    model = ols('mu_freq ~ C(Ethnicity) + Age + Sex', data=df1).fit()
    ancova_results = anova_lm(model, typ=2)
    pp = stats.f_oneway(df1[df1['Ethnicity']=='Korean']['mu_freq'],df1[df1['Ethnicity']=='Japanese']['mu_freq'],
                      df1[df1['Ethnicity']=='Chinese']['mu_freq'],df1[df1['Ethnicity']=='Malay']['mu_freq'],
                      df1[df1['Ethnicity']=='Indian']['mu_freq'], df1[df1['Ethnicity']=='Thai']['mu_freq'])[1]
    p1 = ancova_results['PR(>F)'][0]
    if p1 < 0.1 :
 
        means = df1.groupby('Ethnicity')['mu_freq'].mean()
        sems = df1.groupby('Ethnicity')['mu_freq'].sem()
        max_height = max(means + sems)  # max error-bar height

        # place p-value text slightly above the highest error bar
        text_y = max_height + 0.005  # spacing offset; tweak as needed
        plt.text(x=i-0.25, y=text_y, s='p={:.3f}'.format(p1), fontsize=13)    
    

### Fig. 4E — Per-cell-subset SHM vs age, stratified by sex

In [ ]:
mf = vdata.obs.groupby(['PatientID', 'anno2']).mean()[['mu_freq']]
mf = mf.reset_index()
mf = mf.merge(meta, left_on='PatientID', right_on='PatientID', how='left')
mf = mf[~mf['Ethnicity'].isna()]
mf = mf.dropna()

In [ ]:
for i, a in enumerate(mf.anno2.unique().tolist()):
    isoa = mf[mf.anno2==a]
    g = sns.lmplot(data=isoa, x='Age', y='mu_freq', hue="Sex",                                   
                    hue_order=['Male','Female'],   palette=sex_colors,                      
                   scatter_kws={"ec":"black", "s":50, "alpha":0.6},
                   line_kws={"lw":5, "ls":"--","alpha":1})
    for ax in g.axes.flat:
        for line in ax.lines:
            line.set_path_effects([path_effects.Stroke(linewidth=7, foreground='k'),
                               path_effects.Normal()])
    pv = stats.pearsonr(isoa['Age'], isoa['mu_freq'])
    ap = ols("mu_freq ~ Sex + Ethnicity + Age", data = isoa).fit()
    plt.title('{} \n p={:.3f}, padj={:.3f}'.format(a, pv[1], ap.pvalues[-1]),fontsize=25)
    plt.xlabel('r={:.3f}'.format(pv[0]),fontsize=25)
    plt.ylabel('Mutation frequency',fontsize=25)
    plt.savefig('figures/Fig5_SHM_{}_lmplot.pdf'.format(a), dpi=300, format='pdf',transparent=True, bbox_inches='tight')
    sns.despine()

### Per-isotype SHM vs age (supplementary)

In [ ]:
mf = mbc.obs.groupby(['PatientID', 'c_call_B_VDJ']).mean()[['mu_freq']]
mf = mf.reset_index()
mf = mf.merge(meta, left_on='PatientID', right_on='PatientID', how='left')
mf = mf[~mf['Ethnicity'].isna()]
mf = mf[mf['c_call_B_VDJ']!='IGHE']
mf = mf.dropna()

In [ ]:
for i, a in enumerate(mf.c_call_B_VDJ.unique().tolist()):
    isoa = mf[mf.c_call_B_VDJ==a]
    g = sns.lmplot(data=isoa, x='Age', y='mu_freq', hue="Sex",                                   
                    hue_order=['Male','Female'],   palette=sex_colors,                      
                   scatter_kws={"ec":"black", "s":50, "alpha":0.6},
                   line_kws={"lw":5, "ls":"--","alpha":1})
    for ax in g.axes.flat:
        for line in ax.lines:
            line.set_path_effects([path_effects.Stroke(linewidth=7, foreground='k'),
                               path_effects.Normal()])
    pv = stats.pearsonr(isoa['Age'], isoa['mu_freq'])
    ap = ols("mu_freq ~ Sex + Ethnicity + Age", data = isoa).fit()
    plt.title('{} \n p={:.3f}, padj={:.3f}'.format(a, pv[1], ap.pvalues[-1]),fontsize=25)
    plt.xlabel('r={:.3f}'.format(pv[0]),fontsize=25)
    plt.ylabel('Proportion',fontsize=25)
    sns.despine()

## Fig. 4C / 4F — Isotype distribution and age-associated shifts

In [ ]:
iso_order = ['IGHD','IGHM','IGHG1','IGHG2','IGHG3','IGHG4','IGHA1','IGHA2','IGHE']

In [ ]:
iso_color = ['slategrey','wheat','lightskyblue','dodgerblue','navy','teal','palevioletred','lightsalmon','silver']

In [ ]:
vdata.obs['c_call_B_VDJ'] = vdata.obs['c_call_B_VDJ'].cat.reorder_categories(iso_order)

In [ ]:
vdata.uns['c_call_B_VDJ_colors'] = iso_color


### Fig. 4C — UMAP colored by isotype subclass

In [ ]:
scjp.us(vdata,'c_call_B_VDJ', frameon=False, size=5)
plt.title('')

scjp.save_fig('Fig5_','B_c_call_UMAP',fig_folder='figures')

### Stacked bar of per-subset isotype composition

In [ ]:
vdata.obs['anno2'] = vdata.obs['anno2'].cat.reorder_categories(anno2_order) 

In [ ]:
isoprop = pd.crosstab(vdata.obs['anno2'], vdata.obs['c_call_B_VDJ'], normalize=0)

In [ ]:
g = isoprop.plot(kind='barh', stacked=True, figsize=(6,4), width=0.9, edgecolor='black', linewidth=1,
            color=iso_color) 
plt.ylabel('')
sns.despine()
g.spines['bottom'].set_linewidth(1)
g.spines['left'].set_linewidth(1)

h,l = g.get_legend_handles_labels()
plt.legend(handles= h[::-1], labels = l[::-1],loc=(1.01,0), edgecolor='black')

### Fig. 4F — Per-isotype proportion vs age in memory B cells

In [ ]:
test = vdata[~vdata.obs["BMI"].isna()]

In [ ]:
mbc = test[test.obs['anno1']=='B_Memory']

In [ ]:
mbc = vdata[vdata.obs['anno1']=='B_Memory']

In [ ]:
iso = pd.crosstab(mbc.obs['PatientID'], mbc.obs['c_call_VDJ'], normalize=0) 

In [ ]:
iso = iso.reset_index().melt(id_vars='PatientID')
iso = iso.merge(meta, left_on='PatientID', right_on='PatientID', how='left')
iso = iso[~iso['Ethnicity'].isna()]

In [ ]:
for i, a in enumerate(iso.c_call_VDJ.unique().tolist()):
    isoa = iso[iso.c_call_VDJ==a]
    g = sns.lmplot(data=isoa, x='Age', y='value', hue="Sex",                                   
                    hue_order=['Male','Female'],   palette=sex_colors,                      
                   scatter_kws={"ec":"black", "s":50, "alpha":0.6},
                   line_kws={"lw":5, "ls":"--","alpha":1})
    for ax in g.axes.flat:
        for line in ax.lines:
            line.set_path_effects([path_effects.Stroke(linewidth=7, foreground='k'),
                               path_effects.Normal()])
    pv = stats.pearsonr(isoa['Age'], isoa['value'])
    ap = ols("value ~ Sex + Ethnicity + Age", data = isoa).fit()
    plt.title('{} \n p={:.3f}, padj={:.3f}'.format(a, pv[1], ap.pvalues[-1]),fontsize=25)
    plt.xlabel('r={:.3f}'.format(pv[0]),fontsize=25)
    plt.ylabel('Proportion',fontsize=25)
    plt.savefig('figures/Fig5_isotype_{}_proportion_lmplot.pdf'.format(a), dpi=300, format='pdf',transparent=True, bbox_inches='tight')
    sns.despine()

# Fig. 4G — Ridge regression of BCR features on demographics

We fit Ridge regressions (alpha=1, 1000 bootstraps) for each of the per-donor BCR features:

- **Gini coefficient** of memory B cells (clonality)
- **Average SHM frequency** in memory B cells
- **Per-isotype proportion** within memory B cells

Predictors are age (binarised at 40 y), sex, BMI ≥ 23, and ethnicity dummies. The
combined heatmap concatenates these three blocks of coefficients.

### Ridge regression on memory B cell Gini coefficient

In [ ]:
pf = g0[g0['anno1']=='B_Memory']

In [ ]:
pf = pf[~pf.BMI.isna()]

In [ ]:
 

pf['Ethnicity'] = pf['Ethnicity'].cat.reorder_categories(Ethnicity_order)
coef_dict = {}
pvalue_dict = {}

n_iterations = 1000  # Bootstrap iterations
alpha = 1         # Ridge regularization strength

 
  
pf['Female'] = (pf['Sex'] == 'Female').astype(int)
pf['Older (> 40) '] = (pf['Age'] >= 40).astype(int)

pf['Obesity'] = (pf['BMI'] >= 23).astype(int)


Ethnicity_dummies = pd.get_dummies(pf['Ethnicity'],drop_first=False)

# prepare X and y
X = pd.concat([
    pf[['Older (> 40) ', 'Female', 'Obesity']],
    Ethnicity_dummies
], axis=1)
y = pf['gini_anno1']

# Feature scaling
scaler = standard()
X_scaled = X.copy()

# Fit ridge regression
ridge = Ridge(alpha=alpha)
ridge.fit(X_scaled, y)
coef = pd.Series(ridge.coef_, index=X.columns)
coef_dict['gini'] = coef

# Estimate bootstrap SE and p-values
coef_bootstrap_list = []  # collect in list (avoid DataFrame.append)

for i in range(n_iterations):
    # Resample data
    X_resampled, y_resampled = resample(X_scaled, y)
    ridge.fit(X_resampled, y_resampled)
    coef_bootstrap_list.append(ridge.coef_)

# convert bootstrap results to DataFrame
coef_bootstrap = pd.DataFrame(coef_bootstrap_list, columns=X.columns)

# compute standard error and p-value for each coefficient
coef_se = coef_bootstrap.std()
coef_z = coef / coef_se
coef_p = 2 * (1 - norm.cdf(np.abs(coef_z)))
pvalue_dict[a] = coef_p

# Collect results
codf = pd.DataFrame(coef_dict).T  # coefficients per anno2
pvdf = pd.DataFrame(pvalue_dict).T # p-values per anno2
pvdf.columns = codf.columns
 

In [ ]:
codf_gini = pd.DataFrame(coef_dict).T
pvdf_gini = pd.DataFrame({'gini': coef_p}, index=X.columns).T
pvdf_gini.columns = codf_gini.columns


In [ ]:
# Create annotation matrix for significance asterisks
annot_matrix  = pvdf.applymap(lambda x: '*' if x <= 0.05 else '')

# Plot heatmap
fig, ax = plt.subplots(figsize=(7,0.5)) 
ax = sns.heatmap(codf, annot=annot_matrix, fmt='', cmap='RdBu_r',vmin=-0.026, vmax=0.026,
            linewidths=0.5, linecolor='k',
              cbar_kws={"label": "Ridge Coefficient",
                      "shrink": .5, 'anchor':(-0.3,0), 'aspect':15},)
plt.title('Coefficients from Ridge Regression (p < 0.05)')
plt.xlabel(" ")
plt.ylabel(" ")
cbar = fig.figure.get_children()[-1]
cbar.spines[["bottom", "top", "left","right"]].set_visible(True)
cbar.spines[["bottom", "top", "left","right"]].set_color("k")
 
plt.tight_layout()

plt.savefig('figures/Fig5_MBC_gini_ridge.pdf', dpi=300, format='pdf',transparent=True, bbox_inches='tight')
plt.show()

### Ridge regression on per-isotype proportion in memory B cells

In [ ]:
# 0) Source compositional DataFrame: PatientID x cell-type proportions
pf = pd.crosstab(mbc.obs['PatientID'], mbc.obs['c_call_B_VDJ'], normalize='index')
pf = pf.reset_index().melt(id_vars=['PatientID']).set_index('PatientID') 
pf = pf.merge(meta, left_index=True, right_index=True, how='left')

pf['Ethnicity'] = pf['Ethnicity'].cat.reorder_categories(Ethnicity_order)
coef_dict = {}
pvalue_dict = {}

n_iterations = 1000  # Bootstrap iterations
alpha = 1         # Ridge regularization strength

for a in pf['c_call_B_VDJ'].unique():
    subset = pf[pf['c_call_B_VDJ'] == a].copy()

    # Build dummy variables for categorical predictors
    # keep drop_first=False to inspect all Ethnicity category effects
    subset['Female'] = (subset['Sex'] == 'Female').astype(int)
    subset['Older (> 40) '] = (subset['Age'] >= 40).astype(int)
       
    subset['Obesity'] = (subset['BMI'] >= 23).astype(int)
   
    
    Ethnicity_dummies = pd.get_dummies(subset['Ethnicity'],drop_first=False)

    # prepare X and y
    X = pd.concat([
        subset[['Older (> 40) ', 'Female', 'Obesity']],
        Ethnicity_dummies
    ], axis=1)
    y = subset['value']

    # Feature scaling
    scaler = standard()
    X_scaled = X.copy()

    # Fit ridge regression
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_scaled, y)
    coef = pd.Series(ridge.coef_, index=X.columns)
    coef_dict[a] = coef

    # Estimate bootstrap SE and p-values
    coef_bootstrap_list = []  # collect in list (avoid DataFrame.append)

    for i in range(n_iterations):
        # Resample data
        X_resampled, y_resampled = resample(X_scaled, y)
        ridge.fit(X_resampled, y_resampled)
        coef_bootstrap_list.append(ridge.coef_)

    # convert bootstrap results to DataFrame
    coef_bootstrap = pd.DataFrame(coef_bootstrap_list, columns=X.columns)

    # compute standard error and p-value for each coefficient
    coef_se = coef_bootstrap.std()
    coef_z = coef / coef_se
    coef_p = 2 * (1 - norm.cdf(np.abs(coef_z)))
    pvalue_dict[a] = coef_p

# Collect results
codf = pd.DataFrame(coef_dict).T  # coefficients per anno2
pvdf = pd.DataFrame(pvalue_dict).T # p-values per anno2
pvdf.columns = codf.columns
 

In [ ]:
# Create annotation matrix for significance asterisks
annot_matrix  = pvdf.applymap(lambda x: '*' if x <= 0.05 else '')

# Plot heatmap
fig, ax = plt.subplots(figsize=(7,6)) 
ax = sns.heatmap(codf, annot=annot_matrix, fmt='', cmap='RdBu_r',vmin=-0.026, vmax=0.026,
            linewidths=0.5, linecolor='k',
              cbar_kws={"label": "Ridge Coefficient",
                      "shrink": .5, 'anchor':(-0.3,0), 'aspect':15},)
plt.title('Coefficients from Ridge Regression (p < 0.05)')
plt.xlabel(" ")
plt.ylabel(" ")
cbar = fig.figure.get_children()[-1]
cbar.spines[["bottom", "top", "left","right"]].set_visible(True)
cbar.spines[["bottom", "top", "left","right"]].set_color("k")
 
plt.tight_layout()

plt.savefig('figures/Fig5_B_isotype_ridge.pdf', dpi=300, format='pdf',transparent=True, bbox_inches='tight')
plt.show()

### Ridge regression on average SHM frequency in memory B cells

In [ ]:
mbc = test[test.obs['anno1']=='B_Memory']

In [ ]:
smf = mbc.obs.groupby('PatientID').mean()[['mu_freq']]

In [ ]:
smf = smf.merge(meta, left_index=True, right_index=True, how='left')

In [ ]:
pf = smf.copy()
pf['Ethnicity'] = pf['Ethnicity'].cat.reorder_categories(Ethnicity_order)
coef_dict = {}
pvalue_dict = {}

n_iterations = 1000  # Bootstrap iterations
alpha = 1         # Ridge regularization strength


    # Build dummy variables for categorical predictors
    # keep drop_first=False to inspect all Ethnicity category effects
pf['Female'] = (pf['Sex'] == 'Female').astype(int)
pf['Older (> 40) '] = (pf['Age'] >= pf['Age'].median()).astype(int)

pf['Obesity'] = (pf['BMI'] >= 23).astype(int)
   
    
Ethnicity_dummies = pd.get_dummies(pf['Ethnicity'],drop_first=False)

# prepare X and y
X = pd.concat([
    pf[['Older (> 40) ', 'Female', 'Obesity']],
    Ethnicity_dummies
], axis=1)
y = pf['mu_freq']

# Feature scaling
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X)

# Fit ridge regression
ridge = Ridge(alpha=alpha)
ridge.fit(X, y)
coef = pd.Series(ridge.coef_, index=X.columns)
coef_dict['SHM'] = coef.copy()

# Estimate bootstrap SE and p-values
coef_bootstrap_list = []  # collect in list (avoid DataFrame.append)

for i in range(n_iterations):
    # Resample data
    X_resampled, y_resampled = resample(X, y)
    ridge.fit(X_resampled, y_resampled)
    coef_bootstrap_list.append(ridge.coef_)


In [ ]:

# convert bootstrap results to DataFrame
coef_bootstrap = pd.DataFrame(coef_bootstrap_list, columns=X.columns)

# compute standard error and p-value for each coefficient
coef_se = coef_bootstrap.std()
coef_z = coef / coef_se
coef_p = 2 * (1 - norm.cdf(np.abs(coef_z)))
pvalue_dict[a] = coef_p

# Collect results
codf = pd.DataFrame(coef_dict).T  # coefficients per anno2
pvdf = pd.DataFrame(pvalue_dict).T # p-values per anno2
pvdf.columns = codf.columns


In [ ]:
# Create annotation matrix for significance asterisks
annot_matrix  = pvdf.applymap(lambda x: '*' if x <= 0.05 else '')

# Plot heatmap
fig, ax = plt.subplots(figsize=(7, 0.5)) 
ax = sns.heatmap(codf, annot=annot_matrix, fmt='', cmap='RdBu_r',vmin=-0.026, vmax=0.026,
            linewidths=0.5, linecolor='k',
              cbar_kws={"label": "Ridge Coefficient",
                     'anchor':(-0.3,0), 'aspect':15},)
plt.title('Coefficients from Ridge Regression (p < 0.05)')
plt.xlabel("")
# plt.ylabel("Isotype")
cbar = fig.figure.get_children()[-1]
cbar.spines[["bottom", "top", "left","right"]].set_visible(True)
cbar.spines[["bottom", "top", "left","right"]].set_color("k")
# cbar.set_ylabel('Jaccard Index', size=15)
plt.tight_layout()
plt.savefig('figures/Fig5_B_SHM_ridge.pdf', dpi=300, format='pdf',transparent=True, bbox_inches='tight')
plt.show()

### Combined Fig. 4G heatmap (Gini + SHM + isotype proportions)

In [ ]:
# Build a unified ridge-coefficient table by stacking the three feature blocks above.
# `codf_gini`, `codf_isotype`, `codf_shm` (and matching p-value tables) should already
# be defined from the cells above. Adapt variable names if necessary.
combined_coef = pd.concat([codf_gini, codf_isotype, codf_shm], axis=0)
combined_pval = pd.concat([pvdf_gini, pvdf_isotype, pvdf_shm], axis=0)
annot = combined_pval.applymap(lambda x: '*' if x <= 0.05 else '')

fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(combined_coef, annot=annot, fmt='', cmap='RdBu_r', vmin=-0.026, vmax=0.026,
            linewidths=0.5, linecolor='k',
            cbar_kws={'label': 'Ridge Coefficient', 'shrink': 0.5}, ax=ax)
plt.title('Fig. 4G — BCR features ~ demographics (ridge regression)')
plt.xlabel('')
plt.ylabel('')
cbar = ax.collections[0].colorbar
cbar.outline.set_edgecolor('k')
cbar.outline.set_linewidth(1)
plt.tight_layout()
plt.savefig('figures/Fig4_BCR_features_ridge_combined.pdf', dpi=300, format='pdf', transparent=True, bbox_inches='tight')
plt.show()
